# Task 06 - Modeling baseline voi LightGBM

Bai toan: tai moi session hien tai cua mot user, dua tren thong tin session hien tai va lich su truoc do cua user, du doan user co phat sinh mua hang trong 30 ngay tiep theo hay khong, va neu co thi tong doanh thu trong 30 ngay tiep theo la bao nhieu.

Notebook nay se di theo tung task trong `task_06_modeling.txt`. Phan dau tien chi doc du lieu modeling da tao tu Task 05, loc label window day du, va chon dung feature set V1.

## 0. Setup

In [1]:
import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

from pyspark.sql import SparkSession, functions as F

cwd = Path.cwd().resolve()
if (cwd / "data_pyspark_parquet").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data_pyspark_parquet").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = Path(r"g:/ds")

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

PARQUET_DIR = PROJECT_ROOT / "data_pyspark_parquet"
TRAIN_FEATURE_PATH = PARQUET_DIR / "train_user_session_features_30d"
TEST_FEATURE_PATH = PARQUET_DIR / "test_user_session_features_30d"
FEATURE_VALIDATION_REPORT_JSON_PATH = PARQUET_DIR / "feature_validation_report.json"

MODELING_OUTPUT_DIR = PARQUET_DIR / "modeling_outputs"
MODELING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("week3_task06_modeling_lightgbm_baseline")
    .master("local[*]")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("Project root:", PROJECT_ROOT)
print("Train feature path:", TRAIN_FEATURE_PATH)
print("Test feature path:", TEST_FEATURE_PATH)
print("Modeling output dir:", MODELING_OUTPUT_DIR)
print("PySpark Python:", sys.executable)

Project root: G:\ds
Train feature path: G:\ds\data_pyspark_parquet\train_user_session_features_30d
Test feature path: G:\ds\data_pyspark_parquet\test_user_session_features_30d
Modeling output dir: G:\ds\data_pyspark_parquet\modeling_outputs
PySpark Python: f:\ide\anaconda\python.exe


## 0.1 Feature contract V1

Feature list nay duoc lay tu Task 05. Modeling baseline dung V1 core + 2 derived flags: `is_first_session`, `has_previous_purchase`.

In [2]:
def unique_preserve_order(values):
    seen = set()
    result = []
    for value in values:
        if value not in seen:
            seen.add(value)
            result.append(value)
    return result


KEY_COLUMNS = [
    "fullVisitorId",
    "visit_id",
]

META_COLUMNS = [
    "session_date",
    "visit_start_timestamp",
    "session_year",
    "session_month",
    "has_full_30d_label_window",
]

LABEL_COLUMNS = [
    "future_30d_has_purchase",
    "future_30d_revenue",
]

CURRENT_NUMERIC_TIME_FEATURES_V1 = [
    "visit_number",
    "totals_hits",
    "totals_pageviews",
    "totals_time_on_site",
    "totals_new_visits",
    "is_bounce",
    "session_hour",
    "session_day_of_week",
    "session_month",
]

CURRENT_CATEGORICAL_FEATURES_V1 = [
    "channelGrouping_model",
    "traffic_channel_type_model",
    "device_category_model",
    "geo_country_model",
]

CURRENT_BINARY_FEATURES_V1 = [
    "has_gclid",
    "has_traffic_campaign",
    "has_referral_path",
]

USER_HISTORY_FEATURES_V1 = [
    "user_previous_sessions",
    "user_days_since_first_session",
    "user_days_since_previous_session",
    "user_previous_avg_pageviews",
    "user_previous_avg_time_on_site",
    "user_previous_bounce_rate",
    "user_previous_purchase_count",
    "user_previous_total_revenue",
    "user_days_since_previous_purchase",
]

USER_HISTORY_DERIVED_FLAG_FEATURES_V1 = [
    "is_first_session",
    "has_previous_purchase",
]

FEATURE_SET_V1 = (
    CURRENT_NUMERIC_TIME_FEATURES_V1
    + CURRENT_CATEGORICAL_FEATURES_V1
    + CURRENT_BINARY_FEATURES_V1
    + USER_HISTORY_FEATURES_V1
)

FEATURE_SET_V1_WITH_DERIVED_FLAGS = unique_preserve_order(
    FEATURE_SET_V1 + USER_HISTORY_DERIVED_FLAG_FEATURES_V1
)

CATEGORICAL_FEATURES = CURRENT_CATEGORICAL_FEATURES_V1
BINARY_FEATURES = unique_preserve_order(CURRENT_BINARY_FEATURES_V1 + ["is_bounce"] + USER_HISTORY_DERIVED_FLAG_FEATURES_V1)
NUMERIC_FEATURES = [
    column for column in FEATURE_SET_V1_WITH_DERIVED_FLAGS
    if column not in CATEGORICAL_FEATURES and column not in BINARY_FEATURES
]

MODELING_COLUMNS = unique_preserve_order(
    KEY_COLUMNS + META_COLUMNS + FEATURE_SET_V1_WITH_DERIVED_FLAGS + LABEL_COLUMNS
)

print("Feature count:", len(FEATURE_SET_V1_WITH_DERIVED_FLAGS))
print("Numeric feature count:", len(NUMERIC_FEATURES))
print("Categorical feature count:", len(CATEGORICAL_FEATURES))
print("Binary feature count:", len(BINARY_FEATURES))
print("Selected modeling columns:", len(MODELING_COLUMNS))

Feature count: 27
Numeric feature count: 17
Categorical feature count: 4
Binary feature count: 6
Selected modeling columns: 35


## 1. Doc du lieu modeling

Doc train/test feature table tu Task 05, chi giu cac dong co `has_full_30d_label_window = 1`, va chon key/meta columns, feature set V1, label classification, label revenue.

In [3]:
def assert_required_paths_exist():
    missing_paths = [
        str(path)
        for path in [TRAIN_FEATURE_PATH, TEST_FEATURE_PATH, FEATURE_VALIDATION_REPORT_JSON_PATH]
        if not path.exists()
    ]
    if missing_paths:
        raise FileNotFoundError("Missing required modeling input paths: " + json.dumps(missing_paths, indent=2))


def select_modeling_columns(df, dataset_name):
    missing_columns = [column for column in MODELING_COLUMNS if column not in df.columns]
    if missing_columns:
        raise ValueError(f"{dataset_name} missing modeling columns: {missing_columns}")
    return df.select(*MODELING_COLUMNS)


def read_modeling_dataset(input_path, dataset_name):
    raw_df = spark.read.parquet(str(input_path))
    full_window_df = raw_df.filter(F.col("has_full_30d_label_window") == F.lit(1))
    selected_df = select_modeling_columns(full_window_df, dataset_name)
    return selected_df


assert_required_paths_exist()

with FEATURE_VALIDATION_REPORT_JSON_PATH.open(encoding="utf-8") as file:
    feature_validation_report = json.load(file)

train_modeling_sdf = read_modeling_dataset(TRAIN_FEATURE_PATH, "train")
test_modeling_sdf = read_modeling_dataset(TEST_FEATURE_PATH, "test")

print("Loaded train/test modeling Spark DataFrames")
print("Train columns:", len(train_modeling_sdf.columns))
print("Test columns:", len(test_modeling_sdf.columns))

Loaded train/test modeling Spark DataFrames
Train columns: 35
Test columns: 35


In [60]:
test_modeling_sdf.limit(5).show()

+-------------------+----------+------------+---------------------+------------+-------------+-------------------------+------------+-----------+----------------+-------------------+-----------------+---------+------------+-------------------+---------------------+--------------------------+---------------------+-----------------+---------+--------------------+-----------------+----------------------+-----------------------------+--------------------------------+---------------------------+------------------------------+-------------------------+----------------------------+---------------------------+---------------------------------+----------------+---------------------+-----------------------+------------------+
|      fullVisitorId|  visit_id|session_date|visit_start_timestamp|session_year|session_month|has_full_30d_label_window|visit_number|totals_hits|totals_pageviews|totals_time_on_site|totals_new_visits|is_bounce|session_hour|session_day_of_week|channelGrouping_model|traffic_c

## 1.1 Kiem tra nhanh input sau khi loc label window

In [4]:
def summarize_modeling_input(df, dataset_name):
    row_count = df.count()
    distinct_key_count = df.select(KEY_COLUMNS).distinct().count()
    null_key_rows = df.filter(
        F.col("fullVisitorId").isNull() | F.col("visit_id").isNull()
    ).count()
    label_null_rows = df.filter(
        F.col("future_30d_has_purchase").isNull() | F.col("future_30d_revenue").isNull()
    ).count()

    date_summary = df.agg(
        F.min("session_date").alias("min_session_date"),
        F.max("session_date").alias("max_session_date"),
    ).collect()[0].asDict()

    label_distribution = [
        row.asDict()
        for row in (
            df.groupBy("future_30d_has_purchase")
            .agg(F.count("*").alias("row_count"))
            .withColumn("row_pct", F.col("row_count") / F.lit(row_count) * F.lit(100.0))
            .orderBy("future_30d_has_purchase")
            .collect()
        )
    ]

    revenue_summary = df.agg(
        F.sum(F.when(F.col("future_30d_revenue") > 0, 1).otherwise(0)).alias("positive_revenue_rows"),
        F.min("future_30d_revenue").alias("revenue_min"),
        F.avg("future_30d_revenue").alias("revenue_avg"),
        F.max("future_30d_revenue").alias("revenue_max"),
    ).collect()[0].asDict()

    return {
        "dataset": dataset_name,
        "row_count": row_count,
        "distinct_key_count": distinct_key_count,
        "duplicate_key_rows": row_count - distinct_key_count,
        "null_key_rows": null_key_rows,
        "label_null_rows": label_null_rows,
        "date_summary": date_summary,
        "label_distribution": label_distribution,
        "revenue_summary": revenue_summary,
    }


modeling_input_summary = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "feature_set": "v1_with_derived_flags",
    "feature_count": len(FEATURE_SET_V1_WITH_DERIVED_FLAGS),
    "modeling_columns": MODELING_COLUMNS,
    "train": summarize_modeling_input(train_modeling_sdf, "train"),
    "test": summarize_modeling_input(test_modeling_sdf, "test"),
}

print(json.dumps(modeling_input_summary, indent=2, default=str)[:6000])

{
  "created_at_utc": "2026-06-11T11:26:16.965162+00:00",
  "feature_set": "v1_with_derived_flags",
  "feature_count": 27,
  "modeling_columns": [
    "fullVisitorId",
    "visit_id",
    "session_date",
    "visit_start_timestamp",
    "session_year",
    "session_month",
    "has_full_30d_label_window",
    "visit_number",
    "totals_hits",
    "totals_pageviews",
    "totals_time_on_site",
    "totals_new_visits",
    "is_bounce",
    "session_hour",
    "session_day_of_week",
    "channelGrouping_model",
    "traffic_channel_type_model",
    "device_category_model",
    "geo_country_model",
    "has_gclid",
    "has_traffic_campaign",
    "has_referral_path",
    "user_previous_sessions",
    "user_days_since_first_session",
    "user_days_since_previous_session",
    "user_previous_avg_pageviews",
    "user_previous_avg_time_on_site",
    "user_previous_bounce_rate",
    "user_previous_purchase_count",
    "user_previous_total_revenue",
    "user_days_since_previous_purchase",
  

In [5]:
def assert_task01_modeling_input_ready(summary):
    errors = []
    for dataset_name in ["train", "test"]:
        row = summary[dataset_name]
        if row["row_count"] <= 0:
            errors.append(f"{dataset_name} has no rows after full-window filter")
        if row["duplicate_key_rows"] != 0:
            errors.append(f"{dataset_name} has duplicate key rows: {row['duplicate_key_rows']}")
        if row["null_key_rows"] != 0:
            errors.append(f"{dataset_name} has null key rows: {row['null_key_rows']}")
        if row["label_null_rows"] != 0:
            errors.append(f"{dataset_name} has null labels: {row['label_null_rows']}")

    if train_modeling_sdf.schema != test_modeling_sdf.schema:
        errors.append("Train/test modeling schema mismatch after column selection")

    if errors:
        raise ValueError("\n".join(errors))

    return "Task 01 modeling input ready"


task01_status = assert_task01_modeling_input_ready(modeling_input_summary)
print(task01_status)

Task 01 modeling input ready


## 2. Chia train / validation theo thoi gian

Khong random split. Train split gom cac session truoc `2018-03-01`; validation split gom cac session tu `2018-03-01` den het `2018-03-31`. Test set de nguyen cho final evaluation sau khi da chot model va threshold.

In [7]:
TRAIN_VALIDATION_SPLIT_CONFIG = {
    "split_type": "time_based",
    "train_condition": "session_date < 2018-03-01",
    "validation_condition": "2018-03-01 <= session_date <= 2018-03-31",
    "train_end_exclusive": "2018-03-01",
    "validation_start_inclusive": "2018-03-01",
    "validation_end_inclusive": "2018-03-31",
}

SESSION_DATE_COL = "session_date_for_split"

train_modeling_with_split_date_sdf = train_modeling_sdf.withColumn(
    SESSION_DATE_COL,
    F.to_date(F.col("session_date")),
)

train_split_sdf = train_modeling_with_split_date_sdf.filter(
    F.col(SESSION_DATE_COL) < F.lit(TRAIN_VALIDATION_SPLIT_CONFIG["train_end_exclusive"]).cast("date")
).drop(SESSION_DATE_COL)

validation_sdf = train_modeling_with_split_date_sdf.filter(
    (F.col(SESSION_DATE_COL) >= F.lit(TRAIN_VALIDATION_SPLIT_CONFIG["validation_start_inclusive"]).cast("date"))
    & (F.col(SESSION_DATE_COL) <= F.lit(TRAIN_VALIDATION_SPLIT_CONFIG["validation_end_inclusive"]).cast("date"))
).drop(SESSION_DATE_COL)

print("Created time-based train/validation Spark DataFrames")
print(json.dumps(TRAIN_VALIDATION_SPLIT_CONFIG, indent=2))

Created time-based train/validation Spark DataFrames
{
  "split_type": "time_based",
  "train_condition": "session_date < 2018-03-01",
  "validation_condition": "2018-03-01 <= session_date <= 2018-03-31",
  "train_end_exclusive": "2018-03-01",
  "validation_start_inclusive": "2018-03-01",
  "validation_end_inclusive": "2018-03-31"
}


## 2.1 Kiem tra split theo thoi gian

In [8]:
def summarize_split(df, split_name):
    row_count = df.count()
    distinct_key_count = df.select(KEY_COLUMNS).distinct().count()

    date_summary = df.agg(
        F.min("session_date").alias("min_session_date"),
        F.max("session_date").alias("max_session_date"),
    ).collect()[0].asDict()

    label_distribution = [
        row.asDict()
        for row in (
            df.groupBy("future_30d_has_purchase")
            .agg(F.count("*").alias("row_count"))
            .withColumn("row_pct", F.col("row_count") / F.lit(row_count) * F.lit(100.0))
            .orderBy("future_30d_has_purchase")
            .collect()
        )
    ]

    revenue_summary = df.agg(
        F.sum(F.when(F.col("future_30d_revenue") > 0, 1).otherwise(0)).alias("positive_revenue_rows"),
        F.avg("future_30d_revenue").alias("revenue_avg"),
        F.max("future_30d_revenue").alias("revenue_max"),
    ).collect()[0].asDict()

    return {
        "split": split_name,
        "row_count": row_count,
        "distinct_key_count": distinct_key_count,
        "duplicate_key_rows": row_count - distinct_key_count,
        "date_summary": date_summary,
        "label_distribution": label_distribution,
        "revenue_summary": revenue_summary,
    }


def count_key_overlap(left_df, right_df):
    return left_df.select(KEY_COLUMNS).intersect(right_df.select(KEY_COLUMNS)).count()


split_summary = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "split_config": TRAIN_VALIDATION_SPLIT_CONFIG,
    "train_split": summarize_split(train_split_sdf, "train_split"),
    "validation": summarize_split(validation_sdf, "validation"),
    "train_validation_key_overlap": count_key_overlap(train_split_sdf, validation_sdf),
}

print(json.dumps(split_summary, indent=2, default=str)[:6000])

{
  "created_at_utc": "2026-06-11T11:32:29.425419+00:00",
  "split_config": {
    "split_type": "time_based",
    "train_condition": "session_date < 2018-03-01",
    "validation_condition": "2018-03-01 <= session_date <= 2018-03-31",
    "train_end_exclusive": "2018-03-01",
    "validation_start_inclusive": "2018-03-01",
    "validation_end_inclusive": "2018-03-31"
  },
  "train_split": {
    "split": "train_split",
    "row_count": 1530080,
    "distinct_key_count": 1530080,
    "duplicate_key_rows": 0,
    "date_summary": {
      "min_session_date": "2016-08-01",
      "max_session_date": "2018-02-28"
    },
    "label_distribution": [
      {
        "future_30d_has_purchase": 0,
        "row_count": 1509923,
        "row_pct": 98.6826179023319
      },
      {
        "future_30d_has_purchase": 1,
        "row_count": 20157,
        "row_pct": 1.3173820976680957
      }
    ],
    "revenue_summary": {
      "positive_revenue_rows": 20109,
      "revenue_avg": 5.815708995608056,
   

In [9]:
def assert_task02_time_split_ready(summary):
    errors = []
    train_row = summary["train_split"]
    validation_row = summary["validation"]

    if train_row["row_count"] <= 0:
        errors.append("train_split has no rows")
    if validation_row["row_count"] <= 0:
        errors.append("validation split has no rows")

    if train_row["duplicate_key_rows"] != 0:
        errors.append(f"train_split has duplicate keys: {train_row['duplicate_key_rows']}")
    if validation_row["duplicate_key_rows"] != 0:
        errors.append(f"validation has duplicate keys: {validation_row['duplicate_key_rows']}")
    if summary["train_validation_key_overlap"] != 0:
        errors.append(f"train/validation key overlap: {summary['train_validation_key_overlap']}")

    train_max_date = str(train_row["date_summary"]["max_session_date"])
    validation_min_date = str(validation_row["date_summary"]["min_session_date"])
    validation_max_date = str(validation_row["date_summary"]["max_session_date"])

    if train_max_date >= TRAIN_VALIDATION_SPLIT_CONFIG["train_end_exclusive"]:
        errors.append(f"train_split max date is outside split: {train_max_date}")
    if validation_min_date < TRAIN_VALIDATION_SPLIT_CONFIG["validation_start_inclusive"]:
        errors.append(f"validation min date is outside split: {validation_min_date}")
    if validation_max_date > TRAIN_VALIDATION_SPLIT_CONFIG["validation_end_inclusive"]:
        errors.append(f"validation max date is outside split: {validation_max_date}")

    if errors:
        raise ValueError("\n".join(errors))

    return "Task 02 time split ready"


task02_status = assert_task02_time_split_ready(split_summary)
print(task02_status)

Task 02 time split ready


## 3. Chuan bi du lieu cho LightGBM

LightGBM dung pandas DataFrame. Numeric/binary giu nguyen, categorical convert sang pandas `category`. Category mapping chi fit tu `train_split`; validation va test chi apply category da fit, khong fit rule moi.

In [10]:
import numpy as np
import pandas as pd

CLASSIFICATION_LABEL = "future_30d_has_purchase"
REVENUE_LABEL = "future_30d_revenue"
LOG_REVENUE_LABEL = "log_future_30d_revenue"

USE_DEBUG_NEGATIVE_SAMPLE = False
DEBUG_NEGATIVE_FRACTION = 0.10
DEBUG_RANDOM_SEED = 42

LIGHTGBM_FEATURE_COLUMNS = FEATURE_SET_V1_WITH_DERIVED_FLAGS
PANDAS_MODELING_COLUMNS = unique_preserve_order(
    KEY_COLUMNS + META_COLUMNS + LIGHTGBM_FEATURE_COLUMNS + LABEL_COLUMNS
)

print("LightGBM feature columns:", len(LIGHTGBM_FEATURE_COLUMNS))
print("Pandas modeling columns:", len(PANDAS_MODELING_COLUMNS))
print("Use debug negative sample:", USE_DEBUG_NEGATIVE_SAMPLE)

LightGBM feature columns: 27
Pandas modeling columns: 35
Use debug negative sample: False


## 3.1 Convert Spark splits sang pandas

In [11]:
def maybe_debug_sample_sdf(df, dataset_name):
    if not USE_DEBUG_NEGATIVE_SAMPLE:
        return df

    positive_df = df.filter(F.col(CLASSIFICATION_LABEL) == 1)
    negative_df = df.filter(F.col(CLASSIFICATION_LABEL) == 0).sample(
        withReplacement=False,
        fraction=DEBUG_NEGATIVE_FRACTION,
        seed=DEBUG_RANDOM_SEED,
    )
    sampled_df = positive_df.unionByName(negative_df)
    print(f"Using debug sample for {dataset_name}: all positives + {DEBUG_NEGATIVE_FRACTION:.0%} negatives")
    return sampled_df


def spark_to_pandas_for_modeling(df, dataset_name):
    missing_columns = [column for column in PANDAS_MODELING_COLUMNS if column not in df.columns]
    if missing_columns:
        raise ValueError(f"{dataset_name} missing pandas modeling columns: {missing_columns}")

    selected_df = maybe_debug_sample_sdf(df.select(*PANDAS_MODELING_COLUMNS), dataset_name)
    pdf = selected_df.toPandas()
    print(f"{dataset_name} pandas shape: {pdf.shape}")
    return pdf


train_split_pdf = spark_to_pandas_for_modeling(train_split_sdf, "train_split")
validation_pdf = spark_to_pandas_for_modeling(validation_sdf, "validation")
test_modeling_pdf = spark_to_pandas_for_modeling(test_modeling_sdf, "test")

train_split pandas shape: (1530080, 35)
validation pandas shape: (93998, 35)
test pandas shape: (330036, 35)


## 3.2 Fit category tu train split va transform validation/test

In [12]:
def fit_category_levels(train_pdf, categorical_features):
    category_levels = {}
    for column in categorical_features:
        values = train_pdf[column].dropna().astype(str).unique().tolist()
        category_levels[column] = sorted(values)
    return category_levels


def apply_lightgbm_dtypes(pdf, category_levels, dataset_name):
    result = pdf.copy()

    for column in NUMERIC_FEATURES:
        result[column] = pd.to_numeric(result[column], errors="coerce").astype("float64")

    for column in BINARY_FEATURES:
        result[column] = pd.to_numeric(result[column], errors="coerce").astype("int8")

    for column in CATEGORICAL_FEATURES:
        categories = category_levels[column]
        result[column] = pd.Categorical(result[column].astype(str), categories=categories)

    result[CLASSIFICATION_LABEL] = pd.to_numeric(result[CLASSIFICATION_LABEL], errors="raise").astype("int8")
    result[REVENUE_LABEL] = pd.to_numeric(result[REVENUE_LABEL], errors="raise").astype("float64")
    result[LOG_REVENUE_LABEL] = np.log1p(result[REVENUE_LABEL])

    print(f"Applied LightGBM dtypes for {dataset_name}")
    return result


category_levels = fit_category_levels(train_split_pdf, CATEGORICAL_FEATURES)

train_lgbm_pdf = apply_lightgbm_dtypes(train_split_pdf, category_levels, "train_split")
validation_lgbm_pdf = apply_lightgbm_dtypes(validation_pdf, category_levels, "validation")
test_lgbm_pdf = apply_lightgbm_dtypes(test_modeling_pdf, category_levels, "test")

category_level_summary = {
    column: {
        "category_count": len(levels),
        "sample_categories": levels[:10],
    }
    for column, levels in category_levels.items()
}

print(json.dumps(category_level_summary, indent=2, ensure_ascii=False))

Applied LightGBM dtypes for train_split
Applied LightGBM dtypes for validation
Applied LightGBM dtypes for test
{
  "channelGrouping_model": {
    "category_count": 8,
    "sample_categories": [
      "affiliates",
      "direct",
      "display",
      "organic search",
      "other",
      "paid search",
      "referral",
      "social"
    ]
  },
  "traffic_channel_type_model": {
    "category_count": 5,
    "sample_categories": [
      "direct",
      "organic_search",
      "other",
      "paid_search",
      "referral"
    ]
  },
  "device_category_model": {
    "category_count": 3,
    "sample_categories": [
      "desktop",
      "mobile",
      "tablet"
    ]
  },
  "geo_country_model": {
    "category_count": 127,
    "sample_categories": [
      "albania",
      "algeria",
      "argentina",
      "armenia",
      "australia",
      "austria",
      "azerbaijan",
      "bahrain",
      "bangladesh",
      "belarus"
    ]
  }
}


## 3.3 Tao X/y cho classification va conditional regression

In [13]:
X_train_cls = train_lgbm_pdf[LIGHTGBM_FEATURE_COLUMNS]
y_train_cls = train_lgbm_pdf[CLASSIFICATION_LABEL]

X_valid_cls = validation_lgbm_pdf[LIGHTGBM_FEATURE_COLUMNS]
y_valid_cls = validation_lgbm_pdf[CLASSIFICATION_LABEL]

X_test_cls = test_lgbm_pdf[LIGHTGBM_FEATURE_COLUMNS]
y_test_cls = test_lgbm_pdf[CLASSIFICATION_LABEL]

train_positive_mask = train_lgbm_pdf[REVENUE_LABEL] > 0
validation_positive_mask = validation_lgbm_pdf[REVENUE_LABEL] > 0
test_positive_mask = test_lgbm_pdf[REVENUE_LABEL] > 0

X_train_reg = train_lgbm_pdf.loc[train_positive_mask, LIGHTGBM_FEATURE_COLUMNS]
y_train_reg_log = train_lgbm_pdf.loc[train_positive_mask, LOG_REVENUE_LABEL]

X_valid_reg = validation_lgbm_pdf.loc[validation_positive_mask, LIGHTGBM_FEATURE_COLUMNS]
y_valid_reg_log = validation_lgbm_pdf.loc[validation_positive_mask, LOG_REVENUE_LABEL]
y_valid_reg_revenue = validation_lgbm_pdf.loc[validation_positive_mask, REVENUE_LABEL]

X_test_reg = test_lgbm_pdf.loc[test_positive_mask, LIGHTGBM_FEATURE_COLUMNS]
y_test_reg_log = test_lgbm_pdf.loc[test_positive_mask, LOG_REVENUE_LABEL]
y_test_reg_revenue = test_lgbm_pdf.loc[test_positive_mask, REVENUE_LABEL]

categorical_feature_indices = [
    LIGHTGBM_FEATURE_COLUMNS.index(column)
    for column in CATEGORICAL_FEATURES
]

prepared_data_summary = {
    "feature_count": len(LIGHTGBM_FEATURE_COLUMNS),
    "categorical_features": CATEGORICAL_FEATURES,
    "categorical_feature_indices": categorical_feature_indices,
    "classification": {
        "train_shape": X_train_cls.shape,
        "validation_shape": X_valid_cls.shape,
        "test_shape": X_test_cls.shape,
        "train_positive_count": int(y_train_cls.sum()),
        "validation_positive_count": int(y_valid_cls.sum()),
        "test_positive_count": int(y_test_cls.sum()),
    },
    "regression_positive_revenue_only": {
        "train_shape": X_train_reg.shape,
        "validation_shape": X_valid_reg.shape,
        "test_shape": X_test_reg.shape,
    },
}

print(json.dumps(prepared_data_summary, indent=2, default=str))

{
  "feature_count": 27,
  "categorical_features": [
    "channelGrouping_model",
    "traffic_channel_type_model",
    "device_category_model",
    "geo_country_model"
  ],
  "categorical_feature_indices": [
    9,
    10,
    11,
    12
  ],
  "classification": {
    "train_shape": [
      1530080,
      27
    ],
    "validation_shape": [
      93998,
      27
    ],
    "test_shape": [
      330036,
      27
    ],
    "train_positive_count": 20157,
    "validation_positive_count": 1112,
    "test_positive_count": 5143
  },
  "regression_positive_revenue_only": {
    "train_shape": [
      20109,
      27
    ],
    "validation_shape": [
      1084,
      27
    ],
    "test_shape": [
      4686,
      27
    ]
  }
}


## 3.4 Validate LightGBM input

In [20]:
def assert_lgbm_frame_ready(pdf, dataset_name):
    errors = []
    missing_features = [column for column in LIGHTGBM_FEATURE_COLUMNS if column not in pdf.columns]
    if missing_features:
        errors.append(f"{dataset_name} missing features: {missing_features}")

    for column in CATEGORICAL_FEATURES:
        if not isinstance(pdf[column].dtype, pd.CategoricalDtype):
            errors.append(f"{dataset_name}.{column} is not pandas category")

    label_null_count = int(pdf[[CLASSIFICATION_LABEL, REVENUE_LABEL, LOG_REVENUE_LABEL]].isna().sum().sum())
    if label_null_count != 0:
        errors.append(f"{dataset_name} has label nulls after dtype transform: {label_null_count}")

    numeric_null_counts = pdf[NUMERIC_FEATURES + BINARY_FEATURES].isna().sum()
    bad_numeric_nulls = numeric_null_counts[numeric_null_counts > 0].to_dict()
    if bad_numeric_nulls:
        errors.append(f"{dataset_name} has numeric/binary nulls after dtype transform: {bad_numeric_nulls}")

    if errors:
        raise ValueError("\n".join(errors))

    return {
        "dataset": dataset_name,
        "row_count": len(pdf),
        "feature_count": len(LIGHTGBM_FEATURE_COLUMNS),
        "categorical_null_as_unknown_count": int(pdf[CATEGORICAL_FEATURES].isna().sum().sum()),
    }


task03_validation_summary = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "train": assert_lgbm_frame_ready(train_lgbm_pdf, "train_split"),
    "validation": assert_lgbm_frame_ready(validation_lgbm_pdf, "validation"),
    "test": assert_lgbm_frame_ready(test_lgbm_pdf, "test"),
}

print(json.dumps(task03_validation_summary, indent=2, default=str))
print("Task 03 LightGBM data ready")

{
  "created_at_utc": "2026-06-11T16:34:43.435705+00:00",
  "train": {
    "dataset": "train_split",
    "row_count": 1530080,
    "feature_count": 27,
    "categorical_null_as_unknown_count": 0
  },
  "validation": {
    "dataset": "validation",
    "row_count": 93998,
    "feature_count": 27,
    "categorical_null_as_unknown_count": 0
  },
  "test": {
    "dataset": "test",
    "row_count": 330036,
    "feature_count": 27,
    "categorical_null_as_unknown_count": 0
  }
}
Task 03 LightGBM data ready


## 4. Train Model 1 - Purchase classification

Train `LightGBMClassifier` cho label `future_30d_has_purchase`. Do positive class rat thap, dung `scale_pos_weight = negative_count / positive_count`. Danh gia validation bang accuracy, precision, recall va confusion matrix voi threshold 0.5.

In [21]:
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
)

CLASSIFIER_RANDOM_SEED = 42

positive_count = int((y_train_cls == 1).sum())
negative_count = int((y_train_cls == 0).sum())
scale_pos_weight = negative_count / positive_count

classification_training_summary = {
    "train_rows": int(len(y_train_cls)),
    "positive_count": positive_count,
    "negative_count": negative_count,
    "positive_rate_pct": positive_count / len(y_train_cls) * 100.0,
    "scale_pos_weight": scale_pos_weight,
}

print(json.dumps(classification_training_summary, indent=2, default=str))

{
  "train_rows": 1530080,
  "positive_count": 20157,
  "negative_count": 1509923,
  "positive_rate_pct": 1.3173820976680957,
  "scale_pos_weight": 74.90812124820162
}


In [22]:
purchase_classifier = LGBMClassifier(
    objective="binary",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    random_state=CLASSIFIER_RANDOM_SEED,
    n_jobs=-1,
    verbosity=-1,
)

purchase_classifier.fit(
    X_train_cls,
    y_train_cls,
    eval_set=[(X_valid_cls, y_valid_cls)],
    eval_metric="binary_logloss",
    categorical_feature=CATEGORICAL_FEATURES,
)

valid_purchase_probability_30d = purchase_classifier.predict_proba(X_valid_cls)[:, 1]
train_purchase_probability_30d = purchase_classifier.predict_proba(X_train_cls)[:, 1]
test_purchase_probability_30d = purchase_classifier.predict_proba(X_test_cls)[:, 1]

print("Trained purchase_classifier")

Trained purchase_classifier


## 4.1 Tinh classification metrics

In [23]:
def classification_metrics_at_threshold(y_true, probabilities, threshold):
    predicted = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, predicted)),
        "precision": float(precision_score(y_true, predicted, zero_division=0)),
        "recall": float(recall_score(y_true, predicted, zero_division=0)),
        "confusion_matrix": {
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
        },
        "predicted_positive_count": int(predicted.sum()),
        "predicted_positive_rate_pct": float(predicted.mean() * 100.0),
    }


selected_threshold = 0.5

validation_classification_metrics = {
    "metrics_at_threshold_0_5": classification_metrics_at_threshold(
        y_valid_cls,
        valid_purchase_probability_30d,
        selected_threshold,
    ),
}

print(json.dumps(validation_classification_metrics, indent=2, default=str))

{
  "metrics_at_threshold_0_5": {
    "threshold": 0.5,
    "accuracy": 0.8129641056192685,
    "precision": 0.05059215194018447,
    "recall": 0.8336330935251799,
    "confusion_matrix": {
      "tn": 75490,
      "fp": 17396,
      "fn": 185,
      "tp": 927
    },
    "predicted_positive_count": 18323,
    "predicted_positive_rate_pct": 19.49296793548799
  }
}


## 4.2 Tao classification prediction outputs

In [24]:
CLASSIFICATION_PREDICTION_COLUMNS = unique_preserve_order(
    KEY_COLUMNS + META_COLUMNS + LABEL_COLUMNS
)


def build_classification_predictions(pdf, probabilities, threshold, dataset_name):
    predictions = pdf[CLASSIFICATION_PREDICTION_COLUMNS].copy()
    predictions["dataset"] = dataset_name
    predictions["purchase_probability_30d"] = probabilities
    predictions["selected_threshold"] = threshold
    predictions["predicted_purchase_30d"] = (predictions["purchase_probability_30d"] >= threshold).astype("int8")
    return predictions


train_classification_predictions = build_classification_predictions(
    train_lgbm_pdf,
    train_purchase_probability_30d,
    selected_threshold,
    "train_split",
)
validation_classification_predictions = build_classification_predictions(
    validation_lgbm_pdf,
    valid_purchase_probability_30d,
    selected_threshold,
    "validation",
)
test_classification_predictions = build_classification_predictions(
    test_lgbm_pdf,
    test_purchase_probability_30d,
    selected_threshold,
    "test",
)

classification_prediction_summary = {
    "selected_threshold": selected_threshold,
    "train_prediction_shape": train_classification_predictions.shape,
    "validation_prediction_shape": validation_classification_predictions.shape,
    "test_prediction_shape": test_classification_predictions.shape,
    "validation_predicted_positive_count": int(validation_classification_predictions["predicted_purchase_30d"].sum()),
    "test_predicted_positive_count": int(test_classification_predictions["predicted_purchase_30d"].sum()),
}

print(json.dumps(classification_prediction_summary, indent=2, default=str))
print("Task 04 purchase classification ready")

{
  "selected_threshold": 0.5,
  "train_prediction_shape": [
    1530080,
    13
  ],
  "validation_prediction_shape": [
    93998,
    13
  ],
  "test_prediction_shape": [
    330036,
    13
  ],
  "validation_predicted_positive_count": 18323,
  "test_predicted_positive_count": 74332
}
Task 04 purchase classification ready


## 5. Train Model 2 - Conditional revenue regression

Train `LightGBMRegressor` chi tren cac rows co `future_30d_revenue > 0`. Target la `log1p(future_30d_revenue)`. Prediction output la `predicted_revenue_if_purchase_30d = expm1(predicted_log_revenue)`.

In [29]:
from lightgbm import LGBMRegressor

REGRESSOR_RANDOM_SEED = 42

regression_training_summary = {
    "train_positive_revenue_rows": int(len(X_train_reg)),
    "validation_positive_revenue_rows": int(len(X_valid_reg)),
    "test_positive_revenue_rows": int(len(X_test_reg)),
    "target": LOG_REVENUE_LABEL,
    "target_transform": "log1p(future_30d_revenue)",
}

print(json.dumps(regression_training_summary, indent=2, default=str))

{
  "train_positive_revenue_rows": 20109,
  "validation_positive_revenue_rows": 1084,
  "test_positive_revenue_rows": 4686,
  "target": "log_future_30d_revenue",
  "target_transform": "log1p(future_30d_revenue)"
}


In [30]:
revenue_regressor = LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=REGRESSOR_RANDOM_SEED,
    n_jobs=-1,
    verbosity=-1,
)

revenue_regressor.fit(
    X_train_reg,
    y_train_reg_log,
    eval_set=[(X_valid_reg, y_valid_reg_log)],
    eval_metric="rmse",
    categorical_feature=CATEGORICAL_FEATURES,
)

train_predicted_log_revenue_if_purchase_30d = revenue_regressor.predict(X_train_cls)
valid_predicted_log_revenue_if_purchase_30d = revenue_regressor.predict(X_valid_cls)
test_predicted_log_revenue_if_purchase_30d = revenue_regressor.predict(X_test_cls)

train_predicted_revenue_if_purchase_30d = np.maximum(
    0.0,
    np.expm1(train_predicted_log_revenue_if_purchase_30d),
)
valid_predicted_revenue_if_purchase_30d = np.maximum(
    0.0,
    np.expm1(valid_predicted_log_revenue_if_purchase_30d),
)
test_predicted_revenue_if_purchase_30d = np.maximum(
    0.0,
    np.expm1(test_predicted_log_revenue_if_purchase_30d),
)

print("Trained revenue_regressor")

Trained revenue_regressor


## 5.1 Tinh regression metrics tren positive revenue rows

In [31]:
def regression_metrics_positive_rows(y_true_log, y_true_revenue, predicted_log_all_rows, positive_mask):
    predicted_log_positive = np.asarray(predicted_log_all_rows)[np.asarray(positive_mask)]
    predicted_revenue_positive = np.maximum(0.0, np.expm1(predicted_log_positive))
    y_true_log_array = np.asarray(y_true_log)
    y_true_revenue_array = np.asarray(y_true_revenue)

    log_errors = predicted_log_positive - y_true_log_array
    revenue_errors = predicted_revenue_positive - y_true_revenue_array

    return {
        "positive_row_count": int(len(y_true_revenue_array)),
        "rmse_log_scale": float(np.sqrt(np.mean(np.square(log_errors)))),
        "mae_log_scale": float(np.mean(np.abs(log_errors))),
        "rmse_revenue_scale": float(np.sqrt(np.mean(np.square(revenue_errors)))),
        "mae_revenue_scale": float(np.mean(np.abs(revenue_errors))),
        "actual_revenue_avg": float(np.mean(y_true_revenue_array)),
        "predicted_revenue_avg": float(np.mean(predicted_revenue_positive)),
    }


validation_regression_metrics = regression_metrics_positive_rows(
    y_valid_reg_log,
    y_valid_reg_revenue,
    valid_predicted_log_revenue_if_purchase_30d,
    validation_positive_mask,
)

print(json.dumps(validation_regression_metrics, indent=2, default=str))

{
  "positive_row_count": 1084,
  "rmse_log_scale": 1.0740101790938479,
  "mae_log_scale": 0.8306104266029423,
  "rmse_revenue_scale": 389.6743281527877,
  "mae_revenue_scale": 137.97719761327306,
  "actual_revenue_avg": 169.26162361623616,
  "predicted_revenue_avg": 95.89693809114164
}


## 5.2 Tao regression prediction outputs

In [32]:
REGRESSION_PREDICTION_COLUMNS = unique_preserve_order(
    KEY_COLUMNS + META_COLUMNS + LABEL_COLUMNS
)


def build_regression_predictions(pdf, predicted_log_revenue, predicted_revenue, dataset_name):
    predictions = pdf[REGRESSION_PREDICTION_COLUMNS].copy()
    predictions["dataset"] = dataset_name
    predictions["predicted_log_revenue_if_purchase_30d"] = predicted_log_revenue
    predictions["predicted_revenue_if_purchase_30d"] = predicted_revenue
    return predictions


train_regression_predictions = build_regression_predictions(
    train_lgbm_pdf,
    train_predicted_log_revenue_if_purchase_30d,
    train_predicted_revenue_if_purchase_30d,
    "train_split",
)
validation_regression_predictions = build_regression_predictions(
    validation_lgbm_pdf,
    valid_predicted_log_revenue_if_purchase_30d,
    valid_predicted_revenue_if_purchase_30d,
    "validation",
)
test_regression_predictions = build_regression_predictions(
    test_lgbm_pdf,
    test_predicted_log_revenue_if_purchase_30d,
    test_predicted_revenue_if_purchase_30d,
    "test",
)

regression_prediction_summary = {
    "train_prediction_shape": train_regression_predictions.shape,
    "validation_prediction_shape": validation_regression_predictions.shape,
    "test_prediction_shape": test_regression_predictions.shape,
    "validation_predicted_revenue_if_purchase_avg": float(validation_regression_predictions["predicted_revenue_if_purchase_30d"].mean()),
    "test_predicted_revenue_if_purchase_avg": float(test_regression_predictions["predicted_revenue_if_purchase_30d"].mean()),
}

print(json.dumps(regression_prediction_summary, indent=2, default=str))
print("Task 05 conditional revenue regression ready")

{
  "train_prediction_shape": [
    1530080,
    12
  ],
  "validation_prediction_shape": [
    93998,
    12
  ],
  "test_prediction_shape": [
    330036,
    12
  ],
  "validation_predicted_revenue_if_purchase_avg": 65.73605610646594,
  "test_predicted_revenue_if_purchase_avg": 70.33908378808682
}
Task 05 conditional revenue regression ready


## 5.3 Luu trained models

Luu hai model LightGBM ra dung path trong task. File metadata rieng giu feature contract, category levels va threshold de lan sau load model co the transform data cung format.

In [33]:
import pickle

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

PURCHASE_CLASSIFIER_MODEL_PATH = MODELS_DIR / "lgbm_purchase_classifier_30d.pkl"
REVENUE_REGRESSOR_MODEL_PATH = MODELS_DIR / "lgbm_revenue_regressor_30d.pkl"
MODELING_PREPROCESSING_METADATA_PATH = MODELS_DIR / "lgbm_modeling_preprocessing_30d.json"

with PURCHASE_CLASSIFIER_MODEL_PATH.open("wb") as file:
    pickle.dump(purchase_classifier, file)

with REVENUE_REGRESSOR_MODEL_PATH.open("wb") as file:
    pickle.dump(revenue_regressor, file)

modeling_preprocessing_metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "feature_set": "v1_with_derived_flags",
    "feature_columns": LIGHTGBM_FEATURE_COLUMNS,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "binary_features": BINARY_FEATURES,
    "category_levels_fit_from_train_split": category_levels,
    "classification_label": CLASSIFICATION_LABEL,
    "revenue_label": REVENUE_LABEL,
    "log_revenue_label": LOG_REVENUE_LABEL,
    "selected_threshold": selected_threshold,
    "train_validation_split_config": TRAIN_VALIDATION_SPLIT_CONFIG,
    "classification_metrics_validation": validation_classification_metrics,
    "regression_metrics_validation_positive_rows": validation_regression_metrics,
    "model_paths": {
        "purchase_classifier": str(PURCHASE_CLASSIFIER_MODEL_PATH),
        "revenue_regressor": str(REVENUE_REGRESSOR_MODEL_PATH),
        "preprocessing_metadata": str(MODELING_PREPROCESSING_METADATA_PATH),
    },
}

with MODELING_PREPROCESSING_METADATA_PATH.open("w", encoding="utf-8") as file:
    json.dump(modeling_preprocessing_metadata, file, ensure_ascii=False, indent=2, default=str)

saved_model_summary = {
    "purchase_classifier_model_path": str(PURCHASE_CLASSIFIER_MODEL_PATH),
    "revenue_regressor_model_path": str(REVENUE_REGRESSOR_MODEL_PATH),
    "preprocessing_metadata_path": str(MODELING_PREPROCESSING_METADATA_PATH),
}

print(json.dumps(saved_model_summary, indent=2, ensure_ascii=False))
print("Saved trained LightGBM models")

{
  "purchase_classifier_model_path": "G:\\ds\\models\\lgbm_purchase_classifier_30d.pkl",
  "revenue_regressor_model_path": "G:\\ds\\models\\lgbm_revenue_regressor_30d.pkl",
  "preprocessing_metadata_path": "G:\\ds\\models\\lgbm_modeling_preprocessing_30d.json"
}
Saved trained LightGBM models


## 6. Ket hop thanh expected revenue

Cong thuc cuoi cung: `expected_revenue_30d = purchase_probability_30d * predicted_revenue_if_purchase_30d`. Gia tri nay la doanh thu ky vong trong 30 ngay sau session hien tai.

In [34]:
train_expected_revenue_30d = train_purchase_probability_30d * train_predicted_revenue_if_purchase_30d
valid_expected_revenue_30d = valid_purchase_probability_30d * valid_predicted_revenue_if_purchase_30d
test_expected_revenue_30d = test_purchase_probability_30d * test_predicted_revenue_if_purchase_30d

expected_revenue_summary = {
    "train_expected_revenue_avg": float(np.mean(train_expected_revenue_30d)),
    "validation_expected_revenue_avg": float(np.mean(valid_expected_revenue_30d)),
    "test_expected_revenue_avg": float(np.mean(test_expected_revenue_30d)),
    "train_expected_revenue_max": float(np.max(train_expected_revenue_30d)),
    "validation_expected_revenue_max": float(np.max(valid_expected_revenue_30d)),
    "test_expected_revenue_max": float(np.max(test_expected_revenue_30d)),
}

print(json.dumps(expected_revenue_summary, indent=2, default=str))

{
  "train_expected_revenue_avg": 17.02962561688999,
  "validation_expected_revenue_avg": 16.727842877696997,
  "test_expected_revenue_avg": 18.343892276566777,
  "train_expected_revenue_max": 77575.1310211844,
  "validation_expected_revenue_max": 1347.34705555894,
  "test_expected_revenue_max": 5063.144346889831
}


## 6.1 Evaluate expected revenue tren validation full rows

In [35]:
def revenue_capture_at_k(y_true_revenue, predicted_expected_revenue, k_values=(0.01, 0.05, 0.10)):
    actual = np.asarray(y_true_revenue, dtype="float64")
    predicted = np.asarray(predicted_expected_revenue, dtype="float64")
    order = np.argsort(-predicted)
    total_rows = len(actual)
    total_revenue = float(actual.sum())

    capture = {}
    for k in k_values:
        top_n = max(1, int(np.ceil(total_rows * k)))
        top_indices = order[:top_n]
        captured_revenue = float(actual[top_indices].sum())
        capture[f"top_{int(k * 100)}pct"] = {
            "top_n": int(top_n),
            "captured_revenue": captured_revenue,
            "total_revenue": total_revenue,
            "revenue_capture_pct": float(captured_revenue / total_revenue * 100.0) if total_revenue > 0 else None,
        }
    return capture


def safe_correlation(y_true_revenue, predicted_expected_revenue):
    actual = np.asarray(y_true_revenue, dtype="float64")
    predicted = np.asarray(predicted_expected_revenue, dtype="float64")
    if len(actual) < 2 or np.std(actual) == 0 or np.std(predicted) == 0:
        return None
    return float(np.corrcoef(actual, predicted)[0, 1])


def expected_revenue_metrics(y_true_revenue, predicted_expected_revenue):
    actual = np.asarray(y_true_revenue, dtype="float64")
    predicted = np.asarray(predicted_expected_revenue, dtype="float64")
    errors = predicted - actual
    return {
        "row_count": int(len(actual)),
        "actual_total_revenue": float(actual.sum()),
        "predicted_total_expected_revenue": float(predicted.sum()),
        "actual_avg_revenue": float(actual.mean()),
        "predicted_avg_expected_revenue": float(predicted.mean()),
        "mae": float(np.mean(np.abs(errors))),
        "rmse": float(np.sqrt(np.mean(np.square(errors)))),
        "correlation": safe_correlation(actual, predicted),
        "revenue_capture": revenue_capture_at_k(actual, predicted),
    }


validation_expected_revenue_metrics = expected_revenue_metrics(
    validation_lgbm_pdf[REVENUE_LABEL],
    valid_expected_revenue_30d,
)

print(json.dumps(validation_expected_revenue_metrics, indent=2, default=str))

{
  "row_count": 93998,
  "actual_total_revenue": 183479.6,
  "predicted_total_expected_revenue": 1572383.7748177622,
  "actual_avg_revenue": 1.9519521691950894,
  "predicted_avg_expected_revenue": 16.727842877696997,
  "mae": 17.419526015200255,
  "rmse": 54.57206737095647,
  "correlation": 0.10378905455973339,
  "revenue_capture": {
    "top_1pct": {
      "top_n": 940,
      "captured_revenue": 33426.71,
      "total_revenue": 183479.6,
      "revenue_capture_pct": 18.218216085057957
    },
    "top_5pct": {
      "top_n": 4700,
      "captured_revenue": 102558.59999999999,
      "total_revenue": 183479.6,
      "revenue_capture_pct": 55.8964593338987
    },
    "top_10pct": {
      "top_n": 9400,
      "captured_revenue": 137691.41999999998,
      "total_revenue": 183479.6,
      "revenue_capture_pct": 75.04453901142142
    }
  }
}


## 6.2 Tao expected revenue prediction outputs

In [36]:
EXPECTED_REVENUE_PREDICTION_COLUMNS = unique_preserve_order(
    KEY_COLUMNS + META_COLUMNS + LABEL_COLUMNS
)


def build_expected_revenue_predictions(
    pdf,
    purchase_probability,
    predicted_revenue_if_purchase,
    expected_revenue,
    threshold,
    dataset_name,
):
    predictions = pdf[EXPECTED_REVENUE_PREDICTION_COLUMNS].copy()
    predictions["dataset"] = dataset_name
    predictions["purchase_probability_30d"] = purchase_probability
    predictions["selected_threshold"] = threshold
    predictions["predicted_purchase_30d"] = (predictions["purchase_probability_30d"] >= threshold).astype("int8")
    predictions["predicted_revenue_if_purchase_30d"] = predicted_revenue_if_purchase
    predictions["expected_revenue_30d"] = expected_revenue
    return predictions


train_expected_revenue_predictions = build_expected_revenue_predictions(
    train_lgbm_pdf,
    train_purchase_probability_30d,
    train_predicted_revenue_if_purchase_30d,
    train_expected_revenue_30d,
    selected_threshold,
    "train_split",
)
validation_expected_revenue_predictions = build_expected_revenue_predictions(
    validation_lgbm_pdf,
    valid_purchase_probability_30d,
    valid_predicted_revenue_if_purchase_30d,
    valid_expected_revenue_30d,
    selected_threshold,
    "validation",
)
test_expected_revenue_predictions = build_expected_revenue_predictions(
    test_lgbm_pdf,
    test_purchase_probability_30d,
    test_predicted_revenue_if_purchase_30d,
    test_expected_revenue_30d,
    selected_threshold,
    "test",
)

expected_revenue_prediction_summary = {
    "train_prediction_shape": train_expected_revenue_predictions.shape,
    "validation_prediction_shape": validation_expected_revenue_predictions.shape,
    "test_prediction_shape": test_expected_revenue_predictions.shape,
    "validation_expected_revenue_sum": float(validation_expected_revenue_predictions["expected_revenue_30d"].sum()),
    "test_expected_revenue_sum": float(test_expected_revenue_predictions["expected_revenue_30d"].sum()),
}

print(json.dumps(expected_revenue_prediction_summary, indent=2, default=str))
print("Task 06 expected revenue ready")

{
  "train_prediction_shape": [
    1530080,
    15
  ],
  "validation_prediction_shape": [
    93998,
    15
  ],
  "test_prediction_shape": [
    330036,
    15
  ],
  "validation_expected_revenue_sum": 1572383.7748177622,
  "test_expected_revenue_sum": 6054144.831388993
}
Task 06 expected revenue ready


## 7. Final test evaluation

Chi danh gia tren test sau khi model va threshold da chot tu train/validation. Khong refit model, khong chon lai threshold tren test.

In [37]:
FINAL_TEST_EVALUATION_CONFIG = {
    "uses_refit": False,
    "threshold_source": "validation_fixed_threshold",
    "selected_threshold": selected_threshold,
    "classification_metrics": ["accuracy", "precision", "recall", "confusion_matrix"],
    "regression_metrics_scope": "positive_revenue_rows_only",
    "expected_revenue_metrics_scope": "full_test_rows",
}

print(json.dumps(FINAL_TEST_EVALUATION_CONFIG, indent=2, default=str))

{
  "uses_refit": false,
  "threshold_source": "validation_fixed_threshold",
  "selected_threshold": 0.5,
  "classification_metrics": [
    "accuracy",
    "precision",
    "recall",
    "confusion_matrix"
  ],
  "regression_metrics_scope": "positive_revenue_rows_only",
  "expected_revenue_metrics_scope": "full_test_rows"
}


## 7.1 Classification metrics tren test

In [38]:
test_classification_metrics = {
    "metrics_at_threshold_0_5": classification_metrics_at_threshold(
        y_test_cls,
        test_purchase_probability_30d,
        selected_threshold,
    )
}

print(json.dumps(test_classification_metrics, indent=2, default=str))

{
  "metrics_at_threshold_0_5": {
    "threshold": 0.5,
    "accuracy": 0.7845235065265608,
    "precision": 0.05623419254157025,
    "recall": 0.8127552012444099,
    "confusion_matrix": {
      "tn": 254741,
      "fp": 70152,
      "fn": 963,
      "tp": 4180
    },
    "predicted_positive_count": 74332,
    "predicted_positive_rate_pct": 22.52239149668521
  }
}


## 7.2 Conditional regression metrics tren test positive revenue rows

In [39]:
test_regression_metrics = regression_metrics_positive_rows(
    y_test_reg_log,
    y_test_reg_revenue,
    test_predicted_log_revenue_if_purchase_30d,
    test_positive_mask,
)

print(json.dumps(test_regression_metrics, indent=2, default=str))

{
  "positive_row_count": 4686,
  "rmse_log_scale": 1.1149520838395652,
  "mae_log_scale": 0.8418106807682332,
  "rmse_revenue_scale": 1893.677437010034,
  "mae_revenue_scale": 278.386465017287,
  "actual_revenue_avg": 300.66877080665813,
  "predicted_revenue_avg": 118.72844727790347
}


## 7.3 Expected revenue metrics tren test full rows

In [40]:
test_expected_revenue_metrics = expected_revenue_metrics(
    test_lgbm_pdf[REVENUE_LABEL],
    test_expected_revenue_30d,
)

print(json.dumps(test_expected_revenue_metrics, indent=2, default=str))

{
  "row_count": 330036,
  "actual_total_revenue": 1408933.8599999999,
  "predicted_total_expected_revenue": 6054144.831388993,
  "actual_avg_revenue": 4.269030833000036,
  "predicted_avg_expected_revenue": 18.343892276566777,
  "mae": 20.91716294843877,
  "rmse": 231.0667878503243,
  "correlation": 0.10926049574685803,
  "revenue_capture": {
    "top_1pct": {
      "top_n": 3301,
      "captured_revenue": 751190.56,
      "total_revenue": 1408933.8599999999,
      "revenue_capture_pct": 53.31624012499778
    },
    "top_5pct": {
      "top_n": 16502,
      "captured_revenue": 979469.1699999999,
      "total_revenue": 1408933.8599999999,
      "revenue_capture_pct": 69.51846341459917
    },
    "top_10pct": {
      "top_n": 33004,
      "captured_revenue": 1153778.94,
      "total_revenue": 1408933.8599999999,
      "revenue_capture_pct": 81.8902130721736
    }
  }
}


## 7.4 Tong hop final test metrics

In [41]:
final_test_metrics = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "evaluation_config": FINAL_TEST_EVALUATION_CONFIG,
    "classification": test_classification_metrics,
    "conditional_revenue_regression": test_regression_metrics,
    "expected_revenue": test_expected_revenue_metrics,
}

validation_metrics_for_comparison = {
    "classification": validation_classification_metrics,
    "conditional_revenue_regression": validation_regression_metrics,
    "expected_revenue": validation_expected_revenue_metrics,
}

print(json.dumps(final_test_metrics, indent=2, default=str)[:8000])
print("Task 07 final test evaluation ready")

{
  "created_at_utc": "2026-06-11T17:08:33.478219+00:00",
  "evaluation_config": {
    "uses_refit": false,
    "threshold_source": "validation_fixed_threshold",
    "selected_threshold": 0.5,
    "classification_metrics": [
      "accuracy",
      "precision",
      "recall",
      "confusion_matrix"
    ],
    "regression_metrics_scope": "positive_revenue_rows_only",
    "expected_revenue_metrics_scope": "full_test_rows"
  },
  "classification": {
    "metrics_at_threshold_0_5": {
      "threshold": 0.5,
      "accuracy": 0.7845235065265608,
      "precision": 0.05623419254157025,
      "recall": 0.8127552012444099,
      "confusion_matrix": {
        "tn": 254741,
        "fp": 70152,
        "fn": 963,
        "tp": 4180
      },
      "predicted_positive_count": 74332,
      "predicted_positive_rate_pct": 22.52239149668521
    }
  },
  "conditional_revenue_regression": {
    "positive_row_count": 4686,
    "rmse_log_scale": 1.1149520838395652,
    "mae_log_scale": 0.84181068076823

## 8. Luu output

Ghi predictions ra parquet, metrics va manifest ra JSON. Moi bang prediction gom ca `train_split`, `validation`, `test` trong cung output, phan biet bang cot `dataset`.

In [42]:
CLASSIFICATION_PREDICTIONS_OUTPUT_PATH = MODELING_OUTPUT_DIR / "classification_predictions"
REGRESSION_PREDICTIONS_OUTPUT_PATH = MODELING_OUTPUT_DIR / "regression_predictions"
EXPECTED_REVENUE_PREDICTIONS_OUTPUT_PATH = MODELING_OUTPUT_DIR / "expected_revenue_predictions"
MODELING_METRICS_JSON_PATH = MODELING_OUTPUT_DIR / "modeling_metrics.json"
MODELING_MANIFEST_JSON_PATH = MODELING_OUTPUT_DIR / "modeling_manifest.json"

classification_predictions_all = pd.concat(
    [train_classification_predictions, validation_classification_predictions, test_classification_predictions],
    ignore_index=True,
)
regression_predictions_all = pd.concat(
    [train_regression_predictions, validation_regression_predictions, test_regression_predictions],
    ignore_index=True,
)
expected_revenue_predictions_all = pd.concat(
    [train_expected_revenue_predictions, validation_expected_revenue_predictions, test_expected_revenue_predictions],
    ignore_index=True,
)

print("classification_predictions_all:", classification_predictions_all.shape)
print("regression_predictions_all:", regression_predictions_all.shape)
print("expected_revenue_predictions_all:", expected_revenue_predictions_all.shape)

classification_predictions_all: (1954114, 13)
regression_predictions_all: (1954114, 12)
expected_revenue_predictions_all: (1954114, 15)


In [43]:
def write_pandas_predictions_to_parquet(predictions_pdf, output_path):
    output_path = Path(output_path)
    spark_df = spark.createDataFrame(predictions_pdf)
    (
        spark_df.write
        .mode("overwrite")
        .option("compression", "snappy")
        .parquet(str(output_path))
    )
    return {
        "path": str(output_path),
        "row_count": int(len(predictions_pdf)),
        "column_count": int(len(predictions_pdf.columns)),
        "columns": predictions_pdf.columns.tolist(),
    }


prediction_output_summary = {
    "classification_predictions": write_pandas_predictions_to_parquet(
        classification_predictions_all,
        CLASSIFICATION_PREDICTIONS_OUTPUT_PATH,
    ),
    "regression_predictions": write_pandas_predictions_to_parquet(
        regression_predictions_all,
        REGRESSION_PREDICTIONS_OUTPUT_PATH,
    ),
    "expected_revenue_predictions": write_pandas_predictions_to_parquet(
        expected_revenue_predictions_all,
        EXPECTED_REVENUE_PREDICTIONS_OUTPUT_PATH,
    ),
}

print(json.dumps(prediction_output_summary, indent=2, default=str)[:8000])

{
  "classification_predictions": {
    "path": "G:\\ds\\data_pyspark_parquet\\modeling_outputs\\classification_predictions",
    "row_count": 1954114,
    "column_count": 13,
    "columns": [
      "fullVisitorId",
      "visit_id",
      "session_date",
      "visit_start_timestamp",
      "session_year",
      "session_month",
      "has_full_30d_label_window",
      "future_30d_has_purchase",
      "future_30d_revenue",
      "dataset",
      "purchase_probability_30d",
      "selected_threshold",
      "predicted_purchase_30d"
    ]
  },
  "regression_predictions": {
    "path": "G:\\ds\\data_pyspark_parquet\\modeling_outputs\\regression_predictions",
    "row_count": 1954114,
    "column_count": 12,
    "columns": [
      "fullVisitorId",
      "visit_id",
      "session_date",
      "visit_start_timestamp",
      "session_year",
      "session_month",
      "has_full_30d_label_window",
      "future_30d_has_purchase",
      "future_30d_revenue",
      "dataset",
      "predicted

## 8.1 Luu metrics va manifest

In [44]:
if "PURCHASE_CLASSIFIER_MODEL_PATH" not in globals():
    MODELS_DIR = PROJECT_ROOT / "models"
    PURCHASE_CLASSIFIER_MODEL_PATH = MODELS_DIR / "lgbm_purchase_classifier_30d.pkl"
    REVENUE_REGRESSOR_MODEL_PATH = MODELS_DIR / "lgbm_revenue_regressor_30d.pkl"
    MODELING_PREPROCESSING_METADATA_PATH = MODELS_DIR / "lgbm_modeling_preprocessing_30d.json"

modeling_metrics = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "train_validation_split": TRAIN_VALIDATION_SPLIT_CONFIG,
    "classification_training_summary": classification_training_summary,
    "regression_training_summary": regression_training_summary,
    "validation": validation_metrics_for_comparison,
    "test": final_test_metrics,
}

modeling_manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "task": "week_3_task_06_modeling_lightgbm_30d",
    "problem": "Predict future 30d purchase and expected revenue after current user session.",
    "feature_set": "v1_with_derived_flags",
    "input_paths": {
        "train_features": str(TRAIN_FEATURE_PATH),
        "test_features": str(TEST_FEATURE_PATH),
        "feature_validation_report": str(FEATURE_VALIDATION_REPORT_JSON_PATH),
    },
    "output_paths": {
        "purchase_classifier_model": str(PURCHASE_CLASSIFIER_MODEL_PATH),
        "revenue_regressor_model": str(REVENUE_REGRESSOR_MODEL_PATH),
        "preprocessing_metadata": str(MODELING_PREPROCESSING_METADATA_PATH),
        "classification_predictions": str(CLASSIFICATION_PREDICTIONS_OUTPUT_PATH),
        "regression_predictions": str(REGRESSION_PREDICTIONS_OUTPUT_PATH),
        "expected_revenue_predictions": str(EXPECTED_REVENUE_PREDICTIONS_OUTPUT_PATH),
        "modeling_metrics": str(MODELING_METRICS_JSON_PATH),
        "modeling_manifest": str(MODELING_MANIFEST_JSON_PATH),
    },
    "key_columns": KEY_COLUMNS,
    "meta_columns": META_COLUMNS,
    "label_columns": LABEL_COLUMNS,
    "feature_columns": LIGHTGBM_FEATURE_COLUMNS,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "binary_features": BINARY_FEATURES,
    "category_levels_fit_from_train_split": category_levels,
    "selected_threshold": selected_threshold,
    "train_validation_split": TRAIN_VALIDATION_SPLIT_CONFIG,
    "modeling_input_summary": modeling_input_summary,
    "split_summary": split_summary,
    "prepared_data_summary": prepared_data_summary,
    "prediction_output_summary": prediction_output_summary,
}

with MODELING_METRICS_JSON_PATH.open("w", encoding="utf-8") as file:
    json.dump(modeling_metrics, file, ensure_ascii=False, indent=2, default=str)

with MODELING_MANIFEST_JSON_PATH.open("w", encoding="utf-8") as file:
    json.dump(modeling_manifest, file, ensure_ascii=False, indent=2, default=str)

json_output_summary = {
    "modeling_metrics_json": str(MODELING_METRICS_JSON_PATH),
    "modeling_manifest_json": str(MODELING_MANIFEST_JSON_PATH),
}

print(json.dumps(json_output_summary, indent=2, ensure_ascii=False))

{
  "modeling_metrics_json": "G:\\ds\\data_pyspark_parquet\\modeling_outputs\\modeling_metrics.json",
  "modeling_manifest_json": "G:\\ds\\data_pyspark_parquet\\modeling_outputs\\modeling_manifest.json"
}


## 8.2 Kiem tra output da ghi

In [45]:
def summarize_written_parquet(path):
    df = spark.read.parquet(str(path))
    return {
        "path": str(path),
        "row_count": df.count(),
        "column_count": len(df.columns),
        "datasets": [row["dataset"] for row in df.select("dataset").distinct().orderBy("dataset").collect()],
    }


task08_output_validation = {
    "classification_predictions": summarize_written_parquet(CLASSIFICATION_PREDICTIONS_OUTPUT_PATH),
    "regression_predictions": summarize_written_parquet(REGRESSION_PREDICTIONS_OUTPUT_PATH),
    "expected_revenue_predictions": summarize_written_parquet(EXPECTED_REVENUE_PREDICTIONS_OUTPUT_PATH),
    "modeling_metrics_json_exists": MODELING_METRICS_JSON_PATH.exists(),
    "modeling_manifest_json_exists": MODELING_MANIFEST_JSON_PATH.exists(),
    "purchase_classifier_model_exists": PURCHASE_CLASSIFIER_MODEL_PATH.exists(),
    "revenue_regressor_model_exists": REVENUE_REGRESSOR_MODEL_PATH.exists(),
}

print(json.dumps(task08_output_validation, indent=2, default=str))
print("Task 08 outputs saved")

{
  "classification_predictions": {
    "path": "G:\\ds\\data_pyspark_parquet\\modeling_outputs\\classification_predictions",
    "row_count": 1954114,
    "column_count": 13,
    "datasets": [
      "test",
      "train_split",
      "validation"
    ]
  },
  "regression_predictions": {
    "path": "G:\\ds\\data_pyspark_parquet\\modeling_outputs\\regression_predictions",
    "row_count": 1954114,
    "column_count": 12,
    "datasets": [
      "test",
      "train_split",
      "validation"
    ]
  },
  "expected_revenue_predictions": {
    "path": "G:\\ds\\data_pyspark_parquet\\modeling_outputs\\expected_revenue_predictions",
    "row_count": 1954114,
    "column_count": 15,
    "datasets": [
      "test",
      "train_split",
      "validation"
    ]
  },
  "modeling_metrics_json_exists": true,
  "modeling_manifest_json_exists": true,
  "purchase_classifier_model_exists": true,
  "revenue_regressor_model_exists": true
}
Task 08 outputs saved
